# M22_v2 — MobileNetV2 + SpecAugment on the **corrected** official split**Model ID:** `M22_v2` (augmented) / `M3_v2` (clean control) · **Chunk B** — sound-event backbone**Contributor:** _fill in_ · **Baseline:** M3 (same backbone, no SpecAugment)**Ablation group:** `augmentation_effect`---## Why this run exists`Model_Training_Protocol.md` §1 records that **M2 / M3 / M12 / M22 were all labelled`patient_independent_official_60_40` but actually executed a silent fallback**(`patient_id <= 111 -> test`), giving **11 test patients and 7.1% of cycles**. The labels werecorrected in-place afterwards, but **M22 was never re-run**.So the project's most defensible result — *"SpecAugment gives +0.036 official ICBHI"* — iscurrently measured on 11 patients and 492 cycles. This notebook re-measures it on the realpartition.## What changes vs the original M22**Exactly one thing: the split.** Architecture, optimiser, schedule, class weights, seed,preprocessing and every SpecAugment parameter are copied verbatim from the M22 run so theaugmentation comparison stays one-variable.| | original M22 | this run ||---|---|---|| split | `patient_independent_60_40_patient_id_fallback` | **`official_60_40_patient_independent_corrected`** || test recordings | 492 cycles / 11 patients | **369 recordings / 47 patients** || overlap policy | `drop_from_train` | **reassign leaking patients to train** (protocol §1) |> Protocol §1 mandates the *reassign-to-train* policy (551 train / 369 test = 59.9/40.1). The> original M22 notebook used `drop_from_train`, which keeps the official test set byte-identical> instead. Both are patient-independent; the protocol names the former, so this run uses it and> asserts the exact recording counts.## Run it twice| pass | `CFG["variant"]` | produces ||---|---|---|| 1 | `"augmented"` | `results_M22_v2.json` || 2 | `"clean"` | `results_M3_v2.json` |Both passes share the seed, the split and every hyperparameter, so the difference **is** theaugmentation effect. Section 8 then runs the paired McNemar + bootstrap Δ that Essential #10requires. Without the clean pass there is no defensible Δ — only an absolute number.

## Section 1 — Environment Setup & Dependencies

In [1]:
# ============================================================
# CELL 1 — ENVIRONMENT, DEVICE, REPRODUCIBILITY
# ============================================================
import os, sys, json, math, time, glob, random, shutil, tempfile, datetime, subprocess

def _pip(pkg):
    try:
        __import__(pkg.split("==")[0].replace("-", "_"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

_pip("librosa")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision
import librosa
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, f1_score)
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = False   # speed; seed still fixes init/sampling
torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
PLATFORM = ("Kaggle" if os.path.isdir("/kaggle") else
            "Colab" if os.path.isdir("/content") else "Local")

print(f"platform : {PLATFORM}")
print(f"device   : {DEVICE}  ({GPU_NAME})")
print(f"torch    : {torch.__version__} | torchvision {torchvision.__version__}")
print(f"python   : {sys.version.split()[0]}")

platform : Kaggle
device   : cuda  (Tesla T4)
torch    : 2.10.0+cu128 | torchvision 0.25.0+cu128
python   : 3.12.13


## Section 2 — Configuration & HyperparametersEvery value below is copied from the original M22 run. **Only `split_policy` is new.**

In [2]:
# ============================================================
# CELL 2 — GLOBAL CONFIGURATION
# ============================================================
# ---- THE TWO SWITCHES — set BOTH. They are independent. ----------------------
#
#   VARIANT      what gets trained
#     "augmented"   SpecAugment ON   -> M22_v2  (the paper's best model)
#     "clean"       SpecAugment OFF  -> M3_v2   (its control)
#
#   SPLIT_MODE   which partition it is trained and scored on
#     "corrected"   551/369 recordings, 2,636 test cycles, 47 patients.
#                   Patient-independent: every recording of patients 156 and 218 is
#                   reassigned to train. THIS IS THE PAPER'S HEADLINE PARTITION
#                   (best model 0.5602) and the only one a downstream notebook may
#                   load a checkpoint from.
#     "official"    539/381 recordings, 2,756 test cycles, 49 patients. The published
#                   split file VERBATIM, leak included -- patients 156 and 218 sit on
#                   both sides, so it is NOT patient-independent. Its only legitimate
#                   use is comparison against published ICBHI work, where everyone else
#                   is on the same leaky partition (0.5641 in the comparison table).
#
# "official" does NOT mean "the correct one". It means "the published file, leak and all".
# Getting this wrong produces a checkpoint that looks fine and is not reportable, which is
# why M49 reads the split label out of the checkpoint and refuses the leaky one.
VARIANT    = "augmented"
SPLIT_MODE = "corrected"
# ------------------------------------------------------------------------------

_IDS = {"augmented": ("M22_v2", "MobileNetV2 + SpecAugment (corrected official split)", True),
        "clean":     ("M3_v2",  "MobileNetV2 clean control (corrected official split)", False)}
_mid, _mname, _aug = _IDS[VARIANT]

if SPLIT_MODE not in ("corrected", "official"):
    raise ValueError(f"SPLIT_MODE must be 'corrected' or 'official', got {SPLIT_MODE!r}")
if SPLIT_MODE == "official":
    _mid = _mid + "_official"
    _mname = _mname.replace("corrected official split",
                            "published official split, verbatim")

WORK = "/kaggle/working" if PLATFORM == "Kaggle" else ("/content" if PLATFORM == "Colab" else ".")

CFG = {
    # -- audio preprocessing: protocol section 2 shared defaults --------------
    "sample_rate": 16000, "duration_s": 8.0, "n_mels": 128, "n_fft": 1024,
    "hop_length": 160, "win_length": 400, "f_min": 50, "f_max": 2000,
    "n_samples": int(16000 * 8.0),
    "n_frames": None,                       # -> 801, computed below

    # -- training: M3's swept winner, frozen (see original M22 "NO SWEEP" note) --
    "arch": "mobilenet_v2", "pretrained": True, "freeze_features": False,
    "dropout": 0.3, "lr": 5e-4, "scheduler": "cosine",
    "batch_size": 16, "num_epochs": 40, "weight_decay": 1e-4, "grad_clip": 5.0,
    "use_amp": True, "num_workers": 2, "seed": SEED,
    "imagenet_normalize": True,

    # -- SpecAugment: identical parameters to the original M22 -----------------
    "use_specaugment": _aug,
    "specaug_freq_mask_param": 24, "specaug_time_mask_param": 80,
    "specaug_num_freq_masks": 2, "specaug_num_time_masks": 2,
    "specaug_mask_value": 0.0,

    # -- THE CHANGE ------------------------------------------------------------
    # protocol section 1: reassign every recording of a leaking patient to TRAIN.
    "split_policy": ("reassign_to_train" if SPLIT_MODE == "corrected"
                     else "none_published_verbatim"),
    "split_method": ("official_60_40_patient_independent_corrected" if SPLIT_MODE == "corrected"
                     else "official_60_40_published_verbatim_NOT_patient_independent"),

    # -- task ------------------------------------------------------------------
    "classes": ["Normal", "Crackle", "Wheeze", "Both"], "num_classes": 4,

    # -- identity --------------------------------------------------------------
    "model_id": _mid, "model_name": _mname, "variant": VARIANT,
    "contributor": "OWMTL team",

    # -- runtime ---------------------------------------------------------------
    "use_cache": True,
    "eval_only": False,                     # protocol section 11.C escape hatch
    "n_bootstrap": 1000,                    # Essential #10
    "ckpt_dir":    f"{WORK}/checkpoints_{_mid}",
    "results_dir": f"{WORK}/results_{_mid}",
    "cache_dir":   f"{WORK}/cache",
}
CFG["n_frames"] = 1 + math.floor(CFG["n_samples"] / CFG["hop_length"])   # 801
for d in (CFG["ckpt_dir"], CFG["results_dir"], CFG["cache_dir"]):
    os.makedirs(d, exist_ok=True)

print("=" * 68)
print(f"  {CFG['model_id']}  —  {CFG['model_name']}")
print("=" * 68)
for k in ("arch", "lr", "batch_size", "num_epochs", "use_specaugment",
          "split_method", "split_policy", "seed"):
    print(f"  {k:<18}: {CFG[k]}")
print("=" * 68)
print("=" * 68)
if SPLIT_MODE == "corrected":
    print("  REPORTABLE. Patient-independent, 47 test patients. This is the partition")
    print("  the paper reports and the one M49 requires.")
else:
    print("  *** NOT PATIENT-INDEPENDENT ***")
    print("  Patients 156 and 218 have recordings on both sides of this split, so part")
    print("  of the test set was seen during training. Use this run ONLY to compare")
    print("  against published ICBHI work. Never call it patient-independent, and note")
    print("  that M49 will refuse a checkpoint produced here.")
    print("  For the paper's best model, set SPLIT_MODE = \"corrected\" and re-run.")
print("=" * 68)


  M22_v2_official  —  MobileNetV2 + SpecAugment (published official split, verbatim)
  arch              : mobilenet_v2
  lr                : 0.0005
  batch_size        : 16
  num_epochs        : 40
  use_specaugment   : True
  split_method      : official_60_40_published_verbatim_NOT_patient_independent
  split_policy      : none_published_verbatim
  seed              : 42


In [ ]:
# ============================================================
# OUTPUT SETUP — Google Drive, checked not assumed   [OWMTL_DRIVE_OUTPUT_V2]
# ============================================================
# Spliced from Asif's/engine/drive_setup.py -- do not hand-edit. Re-run
# `python3 "Asif's/engine/patch_colab_v2.py"` to refresh.
#
# On Colab this prompts for Drive access the first time. Accept it: without Drive every
# checkpoint and result lives in /content and disappears when the runtime disconnects, which
# on a 40-epoch run means losing about two hours.
#
# The old block in these notebooks did:
#     try:  drive.mount(...); DRIVE_MOUNTED = True
#     except Exception as e:  print("Falling back to ephemeral /content storage.")
# Two silent failures came out of that: re-mounting an already-mounted Drive raises and the
# except swallowed it, and a mounted-but-unwritable Drive passes makedirs() and only fails
# hours into training. This version PROBES that the directory is genuinely writable and
# RAISES if not. It never silently downgrades.
#
# RUN_MODE controls what happens to an existing OWMTL/<id> folder:
#   "auto"        completed run -> new version (<id>_v2, _v3, ...), old results preserved
#                 interrupted run (checkpoints, no results) -> reuse it so auto-resume works
#   "new_version" always a fresh versioned folder
#   "reuse"       use the folder as-is
#   "overwrite"   delete its contents first  (destructive)
import os, sys, shutil

RUN_MODE = "auto"     # <-- "reuse" to resume an interrupted run in place

import os
import shutil

__all__ = ["setup_output_dir", "mount_drive_if_available"]


def mount_drive_if_available(verbose=True):
    """Return (mounted: bool, detail: str). Never raises."""
    in_colab = "google.colab" in __import__("sys").modules or os.path.exists("/content")
    if not in_colab:
        return False, "not a Colab runtime"

    # Already mounted? Do NOT call mount() again -- that is what raises
    # "Mountpoint must not already contain files".
    if os.path.isdir("/content/drive/MyDrive"):
        if verbose:
            print("Google Drive: already mounted at /content/drive")
        return True, "already mounted"

    try:
        from google.colab import drive
    except Exception as e:
        return False, f"google.colab unavailable ({e})"

    try:
        drive.mount("/content/drive")
        if verbose:
            print("Google Drive: mounted at /content/drive")
        return True, "mounted"
    except Exception as e1:
        # Most common cause: a stale, non-empty mountpoint. force_remount clears it.
        try:
            drive.mount("/content/drive", force_remount=True)
            if verbose:
                print(f"Google Drive: force-remounted (first attempt failed: {e1})")
            return True, "force-remounted"
        except Exception as e2:
            return False, f"mount failed: {e1} | force_remount failed: {e2}"


def _probe_writable(path):
    """Actually write, read back and delete a file. Returns (ok, error)."""
    try:
        os.makedirs(path, exist_ok=True)
        probe = os.path.join(path, ".owmtl_write_probe")
        with open(probe, "w") as f:
            f.write("ok")
        with open(probe) as f:
            if f.read() != "ok":
                return False, "read-back mismatch"
        os.remove(probe)
        return True, None
    except Exception as e:
        return False, str(e)


def _classify(path):
    """'absent' | 'empty' | 'completed' | 'in_progress'."""
    if not os.path.isdir(path):
        return "absent"
    entries = [e for e in os.listdir(path) if not e.startswith(".")]
    if not entries:
        return "empty"
    for dirpath, _d, files in os.walk(path):
        for fn in files:
            if fn.startswith("results_") and fn.endswith(".json"):
                return "completed"
    for dirpath, _d, files in os.walk(path):
        for fn in files:
            if fn.endswith(".pth"):
                return "in_progress"
    return "in_progress"


def _next_version(root, name):
    """name -> name_v2 -> name_v3 ... first path that is absent or empty."""
    v = 2
    while True:
        cand = os.path.join(root, f"{name}_v{v}")
        if _classify(cand) in ("absent", "empty"):
            return cand
        v += 1
        if v > 99:
            raise RuntimeError(f"Refusing to version past {name}_v99 -- clean up {root}.")


def setup_output_dir(model_id, base_name="OWMTL", mode="auto", verbose=True):
    """Resolve a GUARANTEED-WRITABLE output directory.

    Returns (base_dir, info). Raises RuntimeError if nothing writable can be found -- never
    silently downgrades to ephemeral storage.
    """
    if mode not in ("auto", "new_version", "reuse", "overwrite"):
        raise ValueError(f"unknown mode {mode!r}")

    import sys
    in_colab = "google.colab" in sys.modules or os.path.exists("/content")
    mounted, detail = mount_drive_if_available(verbose=verbose) if in_colab else (False, "n/a")

    # Candidate roots, best first. Each is probed before use.
    if in_colab and mounted:
        roots = [(f"/content/drive/MyDrive/{base_name}", "Google Drive (persistent)"),
                 (f"/content/{base_name}", "Colab local (EPHEMERAL)")]
    elif in_colab:
        roots = [(f"/content/{base_name}", "Colab local (EPHEMERAL)")]
    elif os.path.exists("/kaggle/working"):
        roots = [("/kaggle/working", "Kaggle working")]
    else:
        roots = [("./outputs", "local")]

    root = kind = None
    problems = []
    for cand, label in roots:
        ok, err = _probe_writable(cand)
        if ok:
            root, kind = cand, label
            break
        problems.append(f"{cand}: {err}")

    if root is None:
        raise RuntimeError(
            "No writable output directory found. Tried:\n  " + "\n  ".join(problems) +
            "\n\nRefusing to continue: an earlier version silently fell back to ephemeral "
            "storage here, so runs 'succeeded' and then vanished on disconnect.")

    target = root if kind == "Kaggle working" else os.path.join(root, model_id)
    state = _classify(target)

    if mode == "auto":
        if state == "completed":
            base_dir = _next_version(root, model_id)
            action = f"existing run is COMPLETE -> new version {os.path.basename(base_dir)}"
        elif state == "in_progress":
            base_dir, action = target, "existing run is INCOMPLETE -> reusing it (auto-resume)"
        else:
            base_dir, action = target, f"{state} -> using it"
    elif mode == "new_version":
        if state in ("absent", "empty"):
            base_dir, action = target, f"{state} -> using it"
        else:
            base_dir = _next_version(root, model_id)
            action = f"forced new version -> {os.path.basename(base_dir)}"
    elif mode == "overwrite":
        if state not in ("absent", "empty"):
            shutil.rmtree(target, ignore_errors=True)
            action = "OVERWRITE -> previous contents deleted"
        else:
            action = f"{state} -> using it"
        base_dir = target
    else:  # reuse
        base_dir, action = target, f"{state} -> reusing as requested"

    ok, err = _probe_writable(base_dir)
    if not ok:
        raise RuntimeError(f"Chosen directory {base_dir} is not writable: {err}")

    ckpt_dir = os.path.join(base_dir, "checkpoints")
    results_dir = os.path.join(base_dir, "results")
    for d in (ckpt_dir, results_dir):
        os.makedirs(d, exist_ok=True)

    cache_dir = ("/content/owmtl_spec_cache" if in_colab else
                 "/kaggle/working/owmtl_spec_cache" if os.path.exists("/kaggle/working")
                 else "./owmtl_spec_cache")
    os.makedirs(cache_dir, exist_ok=True)

    info = {"base_dir": base_dir, "ckpt_dir": ckpt_dir, "results_dir": results_dir,
            "cache_dir": cache_dir, "storage": kind, "drive_mounted": mounted,
            "drive_detail": detail, "existing_state": state, "action": action, "mode": mode,
            "is_persistent": "EPHEMERAL" not in kind}

    if verbose:
        print("=" * 70)
        print(f"OUTPUT LOCATION  (mode={mode})")
        print("=" * 70)
        print(f"  storage    : {kind}")
        print(f"  existing   : {state}")
        print(f"  decision   : {action}")
        print(f"  base       : {base_dir}")
        print(f"  checkpoints: {ckpt_dir}")
        print(f"  results    : {results_dir}")
        print(f"  spec cache : {cache_dir}   (local disk, disposable)")
        print(f"  write probe: PASSED")
        if not info["is_persistent"]:
            print("\n  *** WARNING: this is EPHEMERAL storage. Everything is lost when the")
            print("      runtime disconnects. Mount Drive before a long run. ***")
        print("=" * 70)
    return base_dir, info

# ---------------------------------------------------------------- resolve
_OWMTL_ID = (CFG.get("model_id") if isinstance(globals().get("CFG"), dict) else None) or "OWMTL"
BASE_DIR, _OUT_INFO = setup_output_dir(_OWMTL_ID, mode=RUN_MODE)
CKPT_DIR    = _OUT_INFO["ckpt_dir"]
RESULTS_DIR = _OUT_INFO["results_dir"]
CACHE_DIR   = _OUT_INFO["cache_dir"]
IN_COLAB      = "google.colab" in sys.modules or os.path.exists("/content")
DRIVE_MOUNTED = _OUT_INFO["drive_mounted"]

# ---------------------------------------------------------------- rewire CFG
# Every downstream write goes through CFG, so redirecting it here is enough -- no other
# cell needs editing. The spectrogram cache deliberately stays on local disk: it is a
# multi-GB disposable memmap and writing it to Drive would be slow and eat the quota.
if isinstance(globals().get("CFG"), dict):
    _redirect = {"ckpt_dir": CKPT_DIR, "results_dir": RESULTS_DIR,
                 "out_dir": RESULTS_DIR, "cache_dir": CACHE_DIR}
    for _k, _v in _redirect.items():
        if _k in CFG:
            CFG[_k] = _v
            os.makedirs(_v, exist_ok=True)
    print("\nCFG redirected to persistent storage:")
    for _k in ("ckpt_dir", "results_dir", "out_dir", "cache_dir"):
        if _k in CFG:
            print(f"  CFG[{_k!r}] = {CFG[_k]}")

if not _OUT_INFO["is_persistent"]:
    print("\nRefusing to fail silently: outputs are EPHEMERAL. Mount Drive and re-run this")
    print("cell before starting a long training run, or accept that results vanish on")
    print("disconnect. (Change RUN_MODE and re-run this cell only -- nothing else changes.)")


## Section 3 — Dataset & the **corrected** patient-independent splitThis is the section the whole notebook exists for.Three hard rules, all enforced by assertion rather than comment:1. **No silent fallback.** If `ICBHI_challenge_train_test.txt` is not found, the cell **raises**.   The original failure was a fallback that ran quietly and mislabelled itself.2. **The file must be the real official split** — 920 recordings, 539/381, 126 patients, with   patients 156 and 218 straddling it. If any of that is wrong, the file is not the official one.3. **After correction: 551 train / 369 test recordings and zero patient overlap.** These are the   numbers `Asif's/audit/official_split.py` reports; if this run disagrees, it stops.

In [ ]:
# ============================================================
# DATA SETUP — credentials, ICBHI audio, official split   [OWMTL_DATA_SETUP_V2]
# ============================================================
# Fully automatic. Nothing to paste, nothing to upload.
#
# ONE-TIME (Colab): left sidebar -> key icon (Secrets) -> add
#     KAGGLE_USERNAME   your kaggle username
#     KAGGLE_KEY        kaggle.com -> Settings -> API -> Create New Token
# Toggle "Notebook access" ON for both. After that every notebook here just works.
#
# The key is never written into the notebook: these files are git-tracked and a Kaggle key is
# full account access. The official split file is EMBEDDED below (3.5 KB gzip+base64), so it
# never needs uploading either.
#
# Sets DATA_ROOT and SPLIT_FILE. Raises with precise instructions if it truly cannot.
import base64 as _b64, glob as _glob, gzip as _gzip, os as _os, subprocess as _sp, sys as _sys

_SPLIT_B64 = "H4sIAAETn2oC/4WcTY/sthFF186vaVUVydLywQiyCAMQsPeDvBcvHuDYDXn+PzKtlnp63vCerA8uWZekqGLpY7ksL8vX5eXL7y9/fXv512//+f66/fnHT6+//fX6t+VgY5sxu+umzF/sq93Yf7+99O+vrz/bL3//6XX79/c/3mA8dXiDbpfLKYynRuesg66DboBuTHVlZ79++2jwMFHfHO6NzhwecG6/vUG/2X+DX/75j59jaf1HuAHspOykHKQcpHwbBAWDrARZCbISZCXISpCVICuFrBSyUshKISuFrBSyUrQVv5q2ssMNYCdlJ+Ug5SClsJLTvemAK+wjK+wjK+wjK+wjK+wjq95Hlst0E72buMGrcnjAjpCaFX3ednUTG9sOXcH5nv8Bdgmvnwf9AxwItyl0vQZOtmnWQddBN0A3XwO3+4zaKU+4AewEBzU7v+yWQjNZ9wUyfgcoFkg74eTOt0NXMGEmE2YyYSYTZjJhJhNmctVpj11guzrhBrATHNTs/Eq3BVKYG7xKaHsK82Wa35xwAzif5R0aNWvUrFGzTs06NavWpLlOjQ2yWIMs1iCLNchiDbJYgyzWKIu1CgYbrdeES9lWUPrtzqSykxNuADspOykHKQcp59nJDp2sOFlxsuJkxcmKkxUnK0FWgqwEWQmyEmQlyEpoK+9bjIIbwE7KTspBykFKsOJkxcmKkxUnK05WnKw4WQmyEmQlyEqQFVhgRheSXQtEey0Q7bVAQNfCfQ5SyrF1mmyngXcaeKexdRpb/woXr9MNwOkG4LSNO23jTtu40zbutEyc9lun/dZpv3XaGP2KYys3xgWyU79nbX2aJLk97X2foL8tzc/zuecQJ9s0G6D70cc7c+jPoT+H/j5P5DPTsQTEEhBLQCwBsYSMxWEeHObBYR5mF9U7U7EEnB5OuElopDRSqjTfCwVUKKAChd4DztPgHRr1adSnUZ9GfTr1iSPk1KeLgvYBRUBVH5edThgHnNe+POkQkXSISDpEJB0ikg4RSYeIpENE0iEi6RCRdIhIOkQkHSKSDhFJh4ikc0LSOSHpnJB0Tkg6JySdE3J63/oIN4CdlJ2Ug5SDlMIKVMIdKuEOlXCHSrhDJdyhEh7PB6wfd4q4wL4Wz+cZAYXy8QhTQbULH7ATxGYhILVFn3ADKPq0c05mEApZEf+X/Vg8urMyPWd/YJtmHdgQzKFNhzZnJ5U7q9Nz+zsTud3JNs066DrohmCpct6Tga4DG9DmPJbnEv+MObCQTK+lBmupwfzNn7N/YJtmHXQddDqWgFgCYgmIJSAWtebpmWqssBms02cUz2zMdOVy32SnbNGHx5NtmnXQddAN0A3QTRfvzhw8OHhw8ODgwcGDgwcHDwEeAjwEeAjwEOAhwENIDw5ryWEtOawXh/XisF4c1ovLeTe4OIsTjKe04scE8YQbwE7KTspBykHKeTpbAmrFJ9wAdlJ2Ug5SDlJKK06z4jQrTrMSNEJBIxQ0QkEjFDRCoUeIii+l6tv0yeatVnjEccJNtzqgx6F6LHoyDyh6LCqlOBn1OEAoQ00Y1aRRTRrVhFFNvRc02rmoDHTATnBQs/MXI0rCOatQdalQdalQdalQdalQdalQdalQdalQdalQdalQdalQdalQdalQdalQdakkPKIuVHoqVF0qVF0qVF0qVF0qVEDaIQyf6UFY6UKi9y7KCvXUssK1Uh/5++er/mSbZh3YgDYH6KZ51c4c4nSI0yFOhzgd4nSIMyDOgDgD4gyIMyDOkHEajItBnAaxGMRiMhZ6dnnA+W2g2nT1foQbwE7KTspBykHK+cVfDd5sqQZvthxwkHKQEgJyCsgpINfDp19BOSEpIdqgaIOiDYo2KNqgaNUNpDpNttNkO7zGdEJSDlIOUkorqZeJQzZ6wE7KTspBSmEl4EZZi36cebBpDexkU12FTwBOuAHspBwI581Stl4pra5UW6xQP9yZfXKyswb5R4P8o0H+0SD/aJB/NMg/GuQfDfKPg3XQddAN0A3QaQ8BHgI8BMQZEGdAnCHjNFgTBmvCYE0YrAmDNWFyTcDHlc2gYnfCDWAnZSflIOUg5XyH32GQlSArQVaCrARZCbISZKWQlUJWClkpZKWQlUJWirZiZMXIilG0RtEaRas/W2zwjLrFtOjyzObbZ0xLLs9sgG6Abn65x7Si8sx0nA5xOsTpEKdDnAFxBsQZEGdAnAFxhoxTPlo72abZAN0A3TyWol+yaUW/ZHOwDroOugG66Us2rUIaUiENqZCGVEhDKqQhFdKQCmlIhTSkQhpSp3eTDwx0A3QDdMqDwTwYzIPBPBjMg8E8GMyDyXlokHo3SL0bpN4NUu8GqTe8d9IarJcG66XBmmiwJhqsCXjPpcF7IA3e52jwPkdruqTWmi6ptYS5TZjbhLlNmNuEuU2Y24T9LGE/S9izEvashD0rYc+6sQpxVoiziuv9YKAboBugUx4M1oTBmjBYEwbzro9O9KjjgPOvWvMCp5G8wGkkL3DgyAscOPICB45czvRg8rQxH99oK2ikVO+9pulXN9OhYH5AMbQBj5gSMqSEDCkhQ0rIkBIypIQMKSuUVk+4AewEBzU7SDk/F2WFInvSGxJZocieFYrsSa9BZIUie1YosmeFIntWKLIfcFCzg5QyWqexdRpbp7F1GlunsXU9tvNPYu5rmiqvmXTVJm0GSZvBCi/IrxdQrgt8VHXCTUL1av1KP61Y6acVK/2XYqX/Uqz064mVfj2xOjyZPOEGsJNyIKRm54tvh+ryfINNXkcnJGUnJfY5SDlIKXyGfkqxhn6Ks8LNb4Wb3wo3vxVufivc/Fa4+a0Vnt+s9OTngOKKzmnZ9N5lTqumz6yDroNugG6AbppxvrF6XaSHel2kh3p/DKcZ6AboBujmHtb7d9zz7WjVX1G9rRD9PdwDbgA7wUHNDlJOL9M7dIrWKVqnaJ2idYrWKdqgaIOiDYo2KNqgaENH6xStU7ROATkF5DogePvnATcJjZRGSielK6X+MapdXL9a+oAbwE5wULODlGLcXb89+oAbwEHKQUoIKCigoICCAgoKKHRARiNkNEJGI2Q0QrDThH7m94AbwK6U7QrNtis0e4PYbCflICgGociS4YNtmnXQddAN0A2hU6WxO9MeDDwYeFBl5BtzUZp+sE0zalP5C1FqfDDQdWAD2pzHUukWUvUXCCcc87uE/qr1wTbNBugG6GZ5oV0aLKcGy0l/1Wrix9HPTMcSEEtALAGxBMQSMhaHOXKYI5fL8M4GtDlAN48z9VuJdoFquC3wU9IHnC7u26+AxVsXJ5t6PJnUObRZJFMf4N6ZAxNzv8AvRW2hTG2hTG2hTG2hTG2hTG2hTG2hTG2hTG2hTG2hTG2Hhfos1GehPgv3KQfBaMqMpsxoVoxmRR9Sl9Angf0HzOrwsRT9E25b9E9bTzamrNEukdBo0iaRcHtcEsJJWUzb2fVzwey90eu88vVgc+GqK8AnFJvBqivAJwSleEZndtGP/u5QlKTNFl3pfsANYCc4dLMYkFGfRn0a9enUp1OfTn2KP+mYwU/uT9gJjjl0vd7N9XVirjcR+EH1yTro+lSn//lzZyb6K+ChQg5ywE5wlmD/D6iHge+6ZgAA"


def _creds():
    """Kaggle credentials, in the order they should already live."""
    try:
        from google.colab import userdata
        u, k = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        if u and k:
            _os.environ["KAGGLE_USERNAME"], _os.environ["KAGGLE_KEY"] = u.strip(), k.strip()
            return "Colab Secrets"
    except Exception:
        pass
    if _os.environ.get("KAGGLE_USERNAME") and _os.environ.get("KAGGLE_KEY"):
        return "environment variables"
    import json as _j
    p = _os.path.expanduser("~/.kaggle/kaggle.json")
    if _os.path.isfile(p):
        try:
            d = _j.load(open(p))
            if d.get("username") and d.get("key"):
                _os.environ["KAGGLE_USERNAME"] = d["username"]
                _os.environ["KAGGLE_KEY"] = d["key"]
                return "~/.kaggle/kaggle.json"
        except Exception:
            pass
    return None


def _is_audio_dir(d):
    return bool(d) and _os.path.isdir(d) and len(_glob.glob(_os.path.join(d, "*.wav"))) >= 900


# Exact paths first — cheap, and they cover every layout this project has actually seen.
_EXACT = [
    "/content/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
    "/content/audio_and_txt_files",
    "/content/drive/MyDrive/OWMTL_data/audio_and_txt_files",
    "/content/drive/MyDrive/ICBHI/audio_and_txt_files",
    "/kaggle/input/respiratory-sound-database/Respiratory_Sound_Database/"
    "Respiratory_Sound_Database/audio_and_txt_files",
    "./data/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files",
]


def _find_audio():
    for d in _EXACT:
        if _is_audio_dir(d):
            return d
    # Glob only roots that are NOT Drive. A recursive walk of a mounted Drive can take
    # minutes and sometimes hangs on a stale FUSE handle.
    for root in ("/kaggle/input", "/content", "./data", "."):
        if not _os.path.isdir(root):
            continue
        for d in sorted(_glob.glob(_os.path.join(root, "**", "audio_and_txt_files"),
                                   recursive=True)):
            if "/drive/" in d.replace("\\", "/"):
                continue
            if _is_audio_dir(d):
                return d
    return None


def _valid_split(p):
    try:
        rows = [l.split() for l in open(p) if len(l.split()) >= 2]
        return len([r for r in rows if r[1].lower() in ("train", "test")]) == 920
    except Exception:
        return False


def _find_split():
    for root in ("/kaggle/input", "/content", ".", _os.path.expanduser("~")):
        if not _os.path.isdir(root):
            continue
        for pat in ("**/ICBHI_challenge_train_test.txt", "**/*train_test*.txt"):
            for p in sorted(_glob.glob(_os.path.join(root, pat), recursive=True)):
                if "/drive/" in p.replace("\\", "/"):
                    continue
                if _valid_split(p):
                    return p
    return None


print("=" * 70)
print("DATA SETUP")
print("=" * 70)

# ---- 1. ICBHI audio --------------------------------------------------------------
DATA_ROOT = _find_audio()
if DATA_ROOT:
    print(f"ICBHI audio : found -> {DATA_ROOT}")
else:
    _src = _creds()
    print(f"ICBHI audio : not present -> downloading  (credentials: {_src or 'NONE'})")
    if not _src:
        raise RuntimeError(
            "ICBHI audio is missing and no Kaggle credentials were found.\n\n"
            "Set them ONCE: Colab left sidebar -> key icon (Secrets) -> add\n"
            "    KAGGLE_USERNAME   your kaggle username\n"
            "    KAGGLE_KEY        kaggle.com -> Settings -> API -> Create New Token\n"
            "Turn 'Notebook access' ON for BOTH, then re-run this cell.\n"
            "You will never be asked again, in this or any other notebook here.")
    _sp.check_call([_sys.executable, "-m", "pip", "install", "-q", "kaggle"])
    _dest = "/content" if _os.path.isdir("/content") else "./data"
    _os.makedirs(_dest, exist_ok=True)
    try:
        _sp.check_call(["kaggle", "datasets", "download", "-d",
                        "vbookshelf/respiratory-sound-database", "-p", _dest, "--unzip"])
    except _sp.CalledProcessError as e:
        raise RuntimeError(
            f"Kaggle download failed (exit {e.returncode}).\n"
            "Most common cause: the Kaggle account has not accepted the dataset's terms.\n"
            "Open https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database "
            "once in a browser while signed in, then re-run this cell.") from e
    DATA_ROOT = _find_audio()
    if not DATA_ROOT:
        raise RuntimeError(f"Download finished but no audio_and_txt_files found under {_dest}")
    print(f"ICBHI audio : ready -> {DATA_ROOT}")

_n_wav = len(_glob.glob(_os.path.join(DATA_ROOT, "*.wav")))
_n_txt = len(_glob.glob(_os.path.join(DATA_ROOT, "*.txt")))
print(f"            {_n_wav} wav / {_n_txt} annotation txt")
assert _n_wav >= 900, f"only {_n_wav} wav files under {DATA_ROOT} — download looks incomplete"

# ---- 2. official split -----------------------------------------------------------
# This notebook REFUSES to fall back to a patient-id rule. That silent fallback gave four
# models an 11-patient test set while labelling itself "official 60/40", and correcting it
# is the whole point of the v2 re-runs (Model_Training_Protocol.md section 1).
SPLIT_FILE = _find_split()
if SPLIT_FILE:
    print(f"Split file  : found -> {SPLIT_FILE}")
else:
    for _cand in (_os.path.join(_os.path.dirname(DATA_ROOT.rstrip("/")),
                                "ICBHI_challenge_train_test.txt"),
                  "/content/ICBHI_challenge_train_test.txt",
                  "./ICBHI_challenge_train_test.txt"):
        try:
            with open(_cand, "wb") as _fh:
                _fh.write(_gzip.decompress(_b64.b64decode(_SPLIT_B64)))
            SPLIT_FILE = _cand
            break
        except Exception:
            continue
    if not SPLIT_FILE:
        raise RuntimeError("could not write the embedded split file anywhere")
    print(f"Split file  : written from embedded copy -> {SPLIT_FILE}")

_rows = [l.split() for l in open(SPLIT_FILE) if len(l.split()) >= 2]
_tr = sum(1 for r in _rows if r[1].lower() == "train")
_te = sum(1 for r in _rows if r[1].lower() == "test")
assert (len(_rows), _tr, _te) == (920, 539, 381), \
    f"this is not the official split file: {len(_rows)} recordings, {_tr} train / {_te} test"
print(f"            verified: 920 recordings, {_tr} train / {_te} test")
print("=" * 70)


In [4]:
# ============================================================
# CELL 4 — BUILD THE CORRECTED SPLIT  (verified, not assumed)
# ============================================================
OFFICIAL_RECORDINGS = 920
OFFICIAL_TRAIN_RECS = 539
OFFICIAL_TEST_RECS  = 381
OFFICIAL_PATIENTS   = 126
OFFICIAL_OVERLAP    = {156, 218}      # straddle the published split
CORRECTED_TRAIN_RECS = 551            # after reassigning the overlap to train
CORRECTED_TEST_RECS  = 369            # = 40.1%

def patient_id_from_stem(stem):
    return int(stem.split("_")[0])

# ---- read the published split verbatim -------------------------------------
split_map = {}
with open(SPLIT_FILE) as fh:
    for line in fh:
        t = line.replace("\t", " ").replace(",", " ").split()
        if len(t) >= 2 and t[1].lower() in ("train", "test"):
            split_map[t[0].replace(".wav", "")] = t[1].lower()

n_tr = sum(1 for v in split_map.values() if v == "train")
n_te = len(split_map) - n_tr
pats = {patient_id_from_stem(s) for s in split_map}
sides = {}
for s, v in split_map.items():
    sides.setdefault(patient_id_from_stem(s), set()).add(v)
overlap = {p for p, v in sides.items() if len(v) > 1}

assert len(split_map) == OFFICIAL_RECORDINGS, f"{len(split_map)} recordings != 920"
assert n_tr == OFFICIAL_TRAIN_RECS and n_te == OFFICIAL_TEST_RECS, f"{n_tr}/{n_te} != 539/381"
assert len(pats) == OFFICIAL_PATIENTS, f"{len(pats)} patients != 126"
assert overlap == OFFICIAL_OVERLAP, f"overlap {sorted(overlap)} != {sorted(OFFICIAL_OVERLAP)}"
print("[OK] file verified as the official ICBHI 2017 split (539/381, 126 patients).")
print(f"     patients {sorted(overlap)} appear on BOTH sides -> not patient-independent.")

# ---- apply the protocol section 1 correction --------------------------------
if SPLIT_MODE == "corrected":
    corrected = {s: ("train" if patient_id_from_stem(s) in overlap else v)
                 for s, v in split_map.items()}
else:
    corrected = dict(split_map)          # published split, verbatim -- no correction

c_tr = sum(1 for v in corrected.values() if v == "train")
c_te = len(corrected) - c_tr
c_sides = {}
for s, v in corrected.items():
    c_sides.setdefault(patient_id_from_stem(s), set()).add(v)
still_leaking = [p for p, v in c_sides.items() if len(v) > 1]

_exp_tr, _exp_te = ((CORRECTED_TRAIN_RECS, CORRECTED_TEST_RECS) if SPLIT_MODE == "corrected"
                    else (OFFICIAL_TRAIN_RECS, OFFICIAL_TEST_RECS))
assert c_tr == _exp_tr, f"train {c_tr} != {_exp_tr}"
assert c_te == _exp_te, f"test  {c_te} != {_exp_te}"
if SPLIT_MODE == "corrected":
    assert not still_leaking, f"patients still on both sides: {still_leaking}"
else:
    assert set(still_leaking) == OFFICIAL_OVERLAP, \
        f"expected {sorted(OFFICIAL_OVERLAP)} to straddle, got {sorted(still_leaking)}"

if SPLIT_MODE == "corrected":
    print(f"[OK] corrected split: {c_tr} train / {c_te} test recordings "
          f"({100 * c_te / len(corrected):.1f}% test), patient-independent.")
else:
    print(f"[OK] PUBLISHED split verbatim: {c_tr} train / {c_te} test recordings "
          f"({100 * c_te / len(corrected):.1f}% test).")
    print("[WARN] this run is NOT patient-independent -- patients 156 and 218 have "
          "recordings on both sides.")
    print("       It exists solely to be directly comparable to published ICBHI work. "
          "Never report it as the project's protocol-compliant result.")
print(f"     moved {n_te - c_te} of {n_te} test recordings into train "
      f"({100 * (n_te - c_te) / n_te:.1f}%).")
print(f"     split_method = {CFG['split_method']}")

[OK] file verified as the official ICBHI 2017 split (539/381, 126 patients).
     patients [156, 218] appear on BOTH sides -> not patient-independent.
[OK] PUBLISHED split verbatim: 539 train / 381 test recordings (41.4% test).
[WARN] this run is NOT patient-independent -- patients 156 and 218 have recordings on both sides.
       It exists solely to be directly comparable to published ICBHI work. Never report it as the project's protocol-compliant result.
     moved 0 of 381 test recordings into train (0.0%).
     split_method = official_60_40_published_verbatim_NOT_patient_independent


In [5]:
# ============================================================
# CELL 5 — CYCLE INDEX  (real audio only — protocol Essential #2)
# ============================================================
LABEL_OF = {(0, 0): 0, (1, 0): 1, (0, 1): 2, (1, 1): 3}   # Normal/Crackle/Wheeze/Both

def parse_annotation(txt_path):
    out = []
    with open(txt_path) as fh:
        for line in fh:
            p = line.split()
            if len(p) >= 4:
                out.append({"start": float(p[0]), "end": float(p[1]),
                            "label": LABEL_OF[(int(p[2]), int(p[3]))]})
    return out

rows, n_missing, n_unlisted, unlisted_stems = [], 0, 0, []
for wav in sorted(glob.glob(os.path.join(DATA_ROOT, "*.wav"))):
    stem = os.path.splitext(os.path.basename(wav))[0]
    txt = os.path.join(DATA_ROOT, stem + ".txt")
    if not os.path.exists(txt):
        n_missing += 1; continue
    sp = corrected.get(stem)
    if sp is None:
        n_unlisted += 1; unlisted_stems.append(stem); continue
    for c in parse_annotation(txt):
        rows.append({"wav_path": wav, "stem": stem, "patient_id": patient_id_from_stem(stem),
                     "start": c["start"], "end": c["end"], "label": c["label"], "split": sp})

df = pd.DataFrame(rows)
assert len(df) > 6000, f"only {len(df)} cycles indexed — check the audio directory"
if n_missing:  print(f"skipped {n_missing} recordings with no annotation .txt")
if n_unlisted:
    print(f"skipped {n_unlisted} recordings absent from the split file: {unlisted_stems}")

df_train = df[df.split == "train"].reset_index(drop=True)
df_test  = df[df.split == "test"].reset_index(drop=True)

# The guarantee, re-checked at cycle level rather than trusted from Cell 4.
leak = set(df_train.patient_id) & set(df_test.patient_id)
if SPLIT_MODE == "corrected":
    assert not leak, f"PATIENT LEAKAGE: {sorted(leak)}"
else:
    assert leak == OFFICIAL_OVERLAP, \
        f"expected exactly {sorted(OFFICIAL_OVERLAP)} on both sides, got {sorted(leak)}"
    _n_leak_cyc = int(df_test.patient_id.isin(leak).sum())
    print(f"[WARN] patients {sorted(leak)} appear in BOTH splits: {_n_leak_cyc} of "
          f"{len(df_test)} test cycles ({100 * _n_leak_cyc / len(df_test):.1f}%) come "
          f"from a patient the model saw in training.")
    print("       This is the published split's own flaw, reproduced deliberately so "
          "the score is comparable to published work. Never report this run as "
          "patient-independent.")

print(f"cycles  : {len(df_train)} train / {len(df_test)} test  "
      f"({100 * len(df_test) / len(df):.1f}% test)")
_ov = "no overlap" if SPLIT_MODE == "corrected" else f"{len(leak)} OVERLAPPING patients"
print(f"patients: {df_train.patient_id.nunique()} train / "
      f"{df_test.patient_id.nunique()} test  [{_ov}]")
print("\nTest-set class distribution:")
for i, name in enumerate(CFG["classes"]):
    n = int((df_test.label == i).sum())
    print(f"  {name:<9} {n:5d}  ({100 * n / len(df_test):5.1f}%)")

print("\n--- contrast with the run this replaces ---")
print(f"  original M22 : 492 test cycles /  11 test patients  (silent fallback)")
print(f"  this run     : {len(df_test):4d} test cycles / {df_test.patient_id.nunique():3d} test patients")

skipped 1 recordings absent from the split file: ['226_1b1_Pl_sc_LittC2SE']
[WARN] patients [156, 218] appear in BOTH splits: 120 of 2756 test cycles (4.4%) come from a patient the model saw in training.
       This is the published split's own flaw, reproduced deliberately so the score is comparable to published work. Never report this run as patient-independent.
cycles  : 4131 train / 2756 test  (40.0% test)
patients: 79 train / 49 test  [2 OVERLAPPING patients]

Test-set class distribution:
  Normal     1579  ( 57.3%)
  Crackle     649  ( 23.5%)
  Wheeze      385  ( 14.0%)
  Both        143  (  5.2%)

--- contrast with the run this replaces ---
  original M22 : 492 test cycles /  11 test patients  (silent fallback)
  this run     : 2756 test cycles /  49 test patients


In [6]:
# ============================================================
# CELL 6 — LOG-MEL EXTRACTION + MEMMAP CACHE
# ============================================================
# Identical to M2/M3/M22: power_to_db(ref=max) then PER-SAMPLE min-max to [0,1],
# short cycles wrap-padded by repetition. Do not "simplify" — the pretrained stem
# was fine-tuned on exactly this scaling.

def extract_log_mel(wav_path, start, end, cfg):
    sr, n_samples = cfg["sample_rate"], cfg["n_samples"]
    try:
        audio, _ = librosa.load(wav_path, sr=sr, offset=start,
                                duration=max(end - start, 0.05), mono=True)
    except Exception as e:
        # A silent all-zero spectrogram here would be trained on and
        # scored as a real cycle. Fail instead of substituting
        # (Model_Training_Protocol.md section 1.2).
        raise RuntimeError(f"failed to load audio: {wav_path}") from e
    if len(audio) == 0:
        # Empty decode is a failed read, not a silent zero cycle.
        raise RuntimeError(f"empty audio decoded from audio: {wav_path}")

    if len(audio) < n_samples:
        audio = np.tile(audio, math.ceil(n_samples / len(audio)))[:n_samples]
    else:
        audio = audio[:n_samples]

    mel = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=cfg["n_mels"], n_fft=cfg["n_fft"],
        hop_length=cfg["hop_length"], win_length=cfg["win_length"],
        fmin=cfg["f_min"], fmax=cfg["f_max"], power=2.0)
    lm = librosa.power_to_db(mel, ref=np.max)
    lm = (lm - lm.min()) / (lm.max() - lm.min() + 1e-8)

    T = lm.shape[1]
    if T < cfg["n_frames"]:
        lm = np.pad(lm, ((0, 0), (0, cfg["n_frames"] - T)), mode="constant")
    else:
        lm = lm[:, :cfg["n_frames"]]
    return lm[None].astype(np.float32)


def build_cache(frame, tag):
    """float16 memmap so DataLoader workers share it without re-decoding audio."""
    path = os.path.join(CFG["cache_dir"], f"specs_{tag}_{len(frame)}.npy")
    shape = (len(frame), 1, CFG["n_mels"], CFG["n_frames"])
    if os.path.exists(path):
        print(f"cache hit  : {os.path.basename(path)}")
        return np.memmap(path, dtype=np.float16, mode="r", shape=shape)
    print(f"building cache: {tag} ({len(frame)} cycles)")
    mm = np.memmap(path, dtype=np.float16, mode="w+", shape=shape)
    for i, r in enumerate(tqdm(frame.itertuples(), total=len(frame), desc=tag)):
        mm[i] = extract_log_mel(r.wav_path, r.start, r.end, CFG).astype(np.float16)
    mm.flush()
    return np.memmap(path, dtype=np.float16, mode="r", shape=shape)

specs_train = build_cache(df_train, "train")
specs_test  = build_cache(df_test, "test")
y_train = df_train.label.values.astype(np.int64)
y_test  = df_test.label.values.astype(np.int64)
print(f"spectrograms: train {specs_train.shape} | test {specs_test.shape}")

building cache: train (4131 cycles)


train:   0%|          | 0/4131 [00:00<?, ?it/s]

building cache: test (2756 cycles)


test:   0%|          | 0/2756 [00:00<?, ?it/s]

spectrograms: train (4131, 1, 128, 801) | test (2756, 1, 128, 801)


In [7]:
# ============================================================
# CELL 7 — SPECAUGMENT, DATASETS, LOADERS, CLASS WEIGHTS
# ============================================================
def spec_augment(x, cfg):
    """x: (1, n_mels, n_frames). Frequency + time masking. Identical to M22."""
    x = x.clone()
    _, n_mels, n_frames = x.shape
    fill = float(cfg["specaug_mask_value"])
    for _ in range(int(cfg["specaug_num_freq_masks"])):
        f = int(torch.randint(0, int(cfg["specaug_freq_mask_param"]) + 1, (1,)).item())
        if 0 < f < n_mels:
            f0 = int(torch.randint(0, n_mels - f + 1, (1,)).item())
            x[:, f0:f0 + f, :] = fill
    for _ in range(int(cfg["specaug_num_time_masks"])):
        t = int(torch.randint(0, int(cfg["specaug_time_mask_param"]) + 1, (1,)).item())
        if 0 < t < n_frames:
            t0 = int(torch.randint(0, n_frames - t + 1, (1,)).item())
            x[:, :, t0:t0 + t] = fill
    return x


class SpecDataset(Dataset):
    def __init__(self, specs, labels, augment=False, cfg=None):
        self.specs, self.labels = specs, labels
        self.augment, self.cfg = bool(augment), cfg
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        x = torch.from_numpy(np.asarray(self.specs[i], dtype=np.float32))
        if self.augment:
            x = spec_augment(x, self.cfg)
        return x, torch.tensor(int(self.labels[i]), dtype=torch.long)


train_ds = SpecDataset(specs_train, y_train, augment=CFG["use_specaugment"], cfg=CFG)
test_ds  = SpecDataset(specs_test,  y_test,  augment=False,                  cfg=CFG)

# Protocol section 7: augmentation is TRAINING-TIME ONLY. Asserted, not assumed.
assert test_ds.augment is False, "SpecAugment must never touch the test split"
print(f"SpecAugment on train: {train_ds.augment}   | on test: {test_ds.augment}")

def make_loader(ds, shuffle):
    return DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle,
                      num_workers=CFG["num_workers"], pin_memory=torch.cuda.is_available(),
                      drop_last=False, persistent_workers=CFG["num_workers"] > 0)

train_loader, test_loader = make_loader(train_ds, True), make_loader(test_ds, False)

# Inverse-frequency weights normalised to mean 1.0 — held constant across M2/M3/M22.
counts = np.maximum(np.bincount(y_train, minlength=CFG["num_classes"]).astype(float), 1.0)
w = counts.sum() / (CFG["num_classes"] * counts)
w = w / w.mean()
class_weights = torch.tensor(w, dtype=torch.float32, device=DEVICE)
print("\ninverse-frequency class weights (train):")
for name, c, wi in zip(CFG["classes"], counts.astype(int), w):
    print(f"  {name:<9} n={c:<6d} weight={wi:.4f}")

SpecAugment on train: True   | on test: False

inverse-frequency class weights (train):
  Normal    n=2057   weight=0.3207
  Crackle   n=1210   weight=0.5452
  Wheeze    n=501    weight=1.3168
  Both      n=363    weight=1.8173


## Section 4 — Model ArchitectureMobileNetV2, ImageNet-pretrained — M3's swept winner, reused verbatim so that *augmentation* is the only variable in the `augmentation_effect` ablation.

In [8]:
# ============================================================
# CELL 8 — MODEL
# ============================================================
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class M3_LightweightCNN(nn.Module):
    """1-channel log-mel -> repeat to 3ch -> ImageNet norm -> MobileNetV2 -> GAP -> logits."""
    def __init__(self, arch="mobilenet_v2", num_classes=4, pretrained=True,
                 freeze_features=False, dropout=0.3, imagenet_normalize=True,
                 n_mels=128, n_frames=801):
        super().__init__()
        weights = "IMAGENET1K_V1" if pretrained else None
        try:
            base = getattr(torchvision.models, arch)(weights=weights)
            self.pretrained_loaded = pretrained
        except Exception as e:
            print(f"[warn] pretrained weights unavailable ({e}); random init")
            base = getattr(torchvision.models, arch)(weights=None)
            self.pretrained_loaded = False
        self.arch = arch
        self.imagenet_normalize = bool(imagenet_normalize and self.pretrained_loaded)
        self.needs_relu = arch.startswith("densenet")
        self.features = base.features
        if freeze_features:
            for p in self.features.parameters():
                p.requires_grad = False
        self.features.eval()
        with torch.no_grad():
            self.feature_dim = int(self.features(torch.zeros(1, 3, n_mels, n_frames)).shape[1])
        self.gap = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.feature_dim, num_classes)

    def forward(self, x):
        x = x.repeat(1, 3, 1, 1)
        if self.imagenet_normalize:
            x = (x - IMAGENET_MEAN.to(x.device)) / IMAGENET_STD.to(x.device)
        f = self.features(x)
        if self.needs_relu:
            f = torch.relu(f)
        return self.classifier(self.dropout(self.gap(f).flatten(1)))


def get_model_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as tmp:
        name = tmp.name
    torch.save(model.state_dict(), name)
    mb = round(os.path.getsize(name) / (1024 * 1024), 2)
    os.remove(name)
    return mb

model = M3_LightweightCNN(
    arch=CFG["arch"], num_classes=CFG["num_classes"], pretrained=CFG["pretrained"],
    freeze_features=CFG["freeze_features"], dropout=CFG["dropout"],
    imagenet_normalize=CFG["imagenet_normalize"],
    n_mels=CFG["n_mels"], n_frames=CFG["n_frames"]).to(DEVICE)

TOTAL_PARAMS = sum(p.numel() for p in model.parameters())
TRAIN_PARAMS = sum(p.numel() for p in model.parameters() if p.requires_grad)
MODEL_SIZE_MB = get_model_size_mb(model)
print(f"{CFG['arch']}: feature_dim={model.feature_dim} | pretrained={model.pretrained_loaded}")
print(f"params: {TOTAL_PARAMS:,} total / {TRAIN_PARAMS:,} trainable | {MODEL_SIZE_MB} MB")

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 135MB/s]


mobilenet_v2: feature_dim=1280 | pretrained=True
params: 2,228,996 total / 2,228,996 trainable | 8.74 MB


## Section 5 — Training LoopPer-epoch checkpointing, auto-resume from `/kaggle/working` **or** `/kaggle/input` (protocol §11.B), scalars cast to native Python types (§11.A), and an `eval_only` escape hatch (§11.C).

In [9]:
# ============================================================
# CELL 9 — METRICS HELPERS
# ============================================================
def icbhi_official(cm):
    """The ICBHI 2017 challenge metric. Se = correct abnormal / all abnormal,
    Sp = correct Normal / all Normal. NOT the macro variant."""
    cm = np.asarray(cm, dtype=float)
    sp = cm[0, 0] / cm[0].sum() if cm[0].sum() else float("nan")
    abn = cm[1:].sum()
    se = (cm[1, 1] + cm[2, 2] + cm[3, 3]) / abn if abn else float("nan")
    return float(se), float(sp), float((se + sp) / 2)


def full_metrics(y_true, y_pred, n_cls=4):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n_cls)))
    p, r, f, sup = precision_recall_fscore_support(
        y_true, y_pred, labels=list(range(n_cls)), zero_division=0)
    spec = []
    for i in range(n_cls):
        tp = cm[i, i]; fp = cm[:, i].sum() - tp; fn = cm[i].sum() - tp
        tn = cm.sum() - tp - fp - fn
        spec.append(tn / (tn + fp) if (tn + fp) else 0.0)
    se_o, sp_o, icbhi_o = icbhi_official(cm)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(p.mean()), "recall_macro": float(r.mean()),
        "f1_macro": float(f.mean()), "specificity_macro": float(np.mean(spec)),
        "icbhi_score": float((r.mean() + np.mean(spec)) / 2),       # macro variant
        "icbhi_score_official": icbhi_o,                             # the reportable one
        "icbhi_se_official": se_o, "icbhi_sp_official": sp_o,
        "per_class": {CFG["classes"][i]: {
            "precision": round(float(p[i]), 4), "recall": round(float(r[i]), 4),
            "f1": round(float(f[i]), 4), "specificity": round(float(spec[i]), 4),
            "support": int(sup[i])} for i in range(n_cls)},
        "confusion_matrix_raw": cm.tolist(),
        "confusion_matrix_normalized": np.round(
            cm / np.maximum(cm.sum(1, keepdims=True), 1), 4).tolist(),
    }

In [10]:
# ============================================================
# CELL 10 — CHECKPOINTS (protocol section 11.A / 11.B)
# ============================================================
def save_checkpoint(path, epoch, model, optimizer, scheduler, best_score, history):
    torch.save({
        "epoch": int(epoch),                    # native ints/floats only — section 11.A
        "best_score": float(best_score),
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "history": history,
        "cfg": {k: v for k, v in CFG.items() if isinstance(v, (int, float, str, bool, type(None)))},
    }, path)


def find_resume_checkpoint():
    """Working dir first, then any attached Kaggle dataset (section 11.B/C)."""
    local = os.path.join(CFG["ckpt_dir"], "latest.pth")
    if os.path.exists(local):
        return local
    for p in sorted(glob.glob(f"/kaggle/input/**/checkpoints_{CFG['model_id']}/latest.pth",
                              recursive=True)) + \
             sorted(glob.glob(f"/kaggle/input/**/{CFG['model_id']}*/*.pth", recursive=True)):
        return p
    return None


start_epoch, best_score, history = 1, -1.0, []
resume = find_resume_checkpoint()
if resume:
    st = torch.load(resume, map_location=DEVICE, weights_only=False)   # trusted, section 11.A
    model.load_state_dict(st["model_state"])
    start_epoch = int(st["epoch"]) + 1
    best_score = float(st["best_score"])
    history = st.get("history", [])
    print(f"resumed from {resume} -> epoch {start_epoch}, best {best_score:.4f}")
else:
    print("no checkpoint found — training from scratch")

if CFG["eval_only"]:
    start_epoch = CFG["num_epochs"] + 1
    print("eval_only=True -> skipping training")

no checkpoint found — training from scratch


In [11]:
# ============================================================
# CELL 11 — TRAIN
# ============================================================
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = (optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CFG["num_epochs"])
             if CFG["scheduler"] == "cosine"
             else optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5))
scaler = torch.amp.GradScaler("cuda", enabled=CFG["use_amp"] and torch.cuda.is_available())

if resume:
    try:
        optimizer.load_state_dict(st["optimizer_state"])
        scheduler.load_state_dict(st["scheduler_state"])
    except Exception as e:
        print(f"[warn] optimiser/scheduler state not restored: {e}")


def run_epoch(loader, train):
    model.train() if train else model.eval()
    tot, n, preds, trues = 0.0, 0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in tqdm(loader, leave=False, desc="train" if train else "eval"):
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
                out = model(x); loss = criterion(out, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
                scaler.step(optimizer); scaler.update()
            tot += float(loss) * y.size(0); n += y.size(0)
            preds.append(out.argmax(1).cpu().numpy()); trues.append(y.cpu().numpy())
    preds, trues = np.concatenate(preds), np.concatenate(trues)
    return tot / max(n, 1), preds, trues


epoch_times = []
for epoch in range(start_epoch, CFG["num_epochs"] + 1):
    t0 = time.time()
    tr_loss, tr_pred, tr_true = run_epoch(train_loader, True)
    va_loss, va_pred, va_true = run_epoch(test_loader, False)
    scheduler.step()
    dt = time.time() - t0; epoch_times.append(dt)

    m = full_metrics(va_true, va_pred)
    history.append({
        "epoch": int(epoch), "train_loss": float(tr_loss), "val_loss": float(va_loss),
        "train_accuracy": float(accuracy_score(tr_true, tr_pred)),
        "val_accuracy": float(m["accuracy"]),
        "train_f1_macro": float(f1_score(tr_true, tr_pred, average="macro", zero_division=0)),
        "val_f1_macro": float(m["f1_macro"]),
        "val_icbhi_score_official": float(m["icbhi_score_official"]),
        "lr": float(optimizer.param_groups[0]["lr"]), "epoch_time_s": float(dt)})

    print(f"ep {epoch:02d}/{CFG['num_epochs']} | train {tr_loss:.4f} val {va_loss:.4f} "
          f"| acc {m['accuracy']:.4f} F1 {m['f1_macro']:.4f} "
          f"| ICBHI-official {m['icbhi_score_official']:.4f} "
          f"(Se {m['icbhi_se_official']:.3f} Sp {m['icbhi_sp_official']:.3f}) | {dt:.0f}s")

    save_checkpoint(os.path.join(CFG["ckpt_dir"], "latest.pth"),
                    epoch, model, optimizer, scheduler, best_score, history)
    if m["icbhi_score_official"] > best_score:      # primary metric = the OFFICIAL one
        best_score = m["icbhi_score_official"]
        save_checkpoint(os.path.join(CFG["ckpt_dir"], "best_model.pth"),
                        epoch, model, optimizer, scheduler, best_score, history)
        print(f"    -> new best ICBHI-official {best_score:.4f}, checkpoint saved")

print(f"\ntraining done. best ICBHI-official = {best_score:.4f}")

train:   0%|          | 0/259 [00:00<?, ?it/s]

/tmp/ipykernel_119/3428382995.py:34: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  tot += float(loss) * y.size(0); n += y.size(0)


eval:   0%|          | 0/173 [00:00<?, ?it/s]

ep 01/40 | train 1.2927 val 1.9947 | acc 0.3904 F1 0.2994 | ICBHI-official 0.3986 (Se 0.455 Sp 0.343) | 82s
    -> new best ICBHI-official 0.3986, checkpoint saved


train:   0%|          | 0/259 [00:00<?, ?it/s]

...

eval:   0%|          | 0/173 [00:00<?, ?it/s]

ep 39/40 | train 0.1701 val 2.9776 | acc 0.5635 F1 0.4223 | ICBHI-official 0.5480 (Se 0.442 Sp 0.654) | 20s


train:   0%|          | 0/259 [00:00<?, ?it/s]

eval:   0%|          | 0/173 [00:00<?, ?it/s]

ep 40/40 | train 0.1800 val 2.9676 | acc 0.5464 F1 0.4232 | ICBHI-official 0.5365 (Se 0.468 Sp 0.605) | 20s

training done. best ICBHI-official = 0.5641


## Section 6 — Evaluation, Confidence Interval & PlotsEssential #10 requires a bootstrap 95% CI on the headline metric. The resampling unit is the**patient**, not the cycle: cycles from one patient are not independent, and resampling themindividually would produce an interval far narrower than the evidence supports.

In [12]:
# ============================================================
# CELL 12 — FINAL EVALUATION ON THE BEST CHECKPOINT
# ============================================================
best_path = os.path.join(CFG["ckpt_dir"], "best_model.pth")
if os.path.exists(best_path):
    st = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(st["model_state"])
    best_epoch = int(st["epoch"])
    history = st.get("history", history)
    print(f"loaded best checkpoint (epoch {best_epoch})")
else:
    best_epoch = int(history[-1]["epoch"]) if history else 0
    print("[warn] no best_model.pth — evaluating the current weights")

_, y_pred, y_true = run_epoch(test_loader, False)
final = full_metrics(y_true, y_pred)

# ---- bootstrap CI, resampling PATIENTS -------------------------------------
test_pids = df_test.patient_id.values
uniq = np.unique(test_pids)
idx_by_pid = {p: np.flatnonzero(test_pids == p) for p in uniq}
rng = np.random.default_rng(SEED)
boot = []
for _ in range(CFG["n_bootstrap"]):
    pick = np.concatenate([idx_by_pid[p] for p in rng.choice(uniq, len(uniq), replace=True)])
    cm = confusion_matrix(y_true[pick], y_pred[pick], labels=list(range(4)))
    v = icbhi_official(cm)[2]
    if v == v:
        boot.append(v)
ci = [round(float(x), 4) for x in np.percentile(boot, [2.5, 97.5])]
final["icbhi_score_official_ci95"] = ci
final["ci_method"] = f"patient-level bootstrap, B={len(boot)}, {len(uniq)} test patients"

# ---- inference latency ------------------------------------------------------
model.eval()
dummy = torch.randn(1, 1, CFG["n_mels"], CFG["n_frames"], device=DEVICE)
with torch.no_grad():
    for _ in range(10): model(dummy)
    ts = []
    for _ in range(50):
        t0 = time.time(); model(dummy)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        ts.append((time.time() - t0) * 1000)
inference_ms = float(np.median(ts))

print("=" * 68)
print(f"{CFG['model_id']} FINAL — corrected official split")
print("=" * 68)
print(f"  accuracy              {final['accuracy']:.4f}")
print(f"  macro F1              {final['f1_macro']:.4f}")
print(f"  ICBHI (macro variant) {final['icbhi_score']:.4f}   <- NOT reportable")
print(f"  ICBHI OFFICIAL        {final['icbhi_score_official']:.4f}  CI95 {ci}")
print(f"     Se {final['icbhi_se_official']:.4f} | Sp {final['icbhi_sp_official']:.4f}")
print(f"  inference             {inference_ms:.2f} ms/sample")
print("=" * 68)
print("  reference points")
print("    original M22 (11-patient fallback split) : 0.6495")
print("    M3  clean, official patient-disjoint     : 0.4490")
print("    M2  clean, official patient-disjoint     : 0.4139")
print("    published ICBHI SOTA, official 60/40     : ~0.60-0.65")
print("=" * 68)

# ---- per-cycle prediction dump: enables the paired test in Section 8 --------
pred_csv = os.path.join(CFG["results_dir"], f"preds_{CFG['model_id']}.csv")
pd.DataFrame({"stem": df_test.stem, "patient_id": test_pids,
              "y_true": y_true, "y_pred": y_pred}).to_csv(pred_csv, index=False)
print(f"per-cycle predictions -> {pred_csv}")

loaded best checkpoint (epoch 26)


eval:   0%|          | 0/173 [00:00<?, ?it/s]

M22_v2_official FINAL — corrected official split
  accuracy              0.5798
  macro F1              0.4459
  ICBHI (macro variant) 0.6358   <- NOT reportable
  ICBHI OFFICIAL        0.5641  CI95 [0.5121, 0.6144]
     Se 0.4562 | Sp 0.6719
  inference             5.36 ms/sample
  reference points
    original M22 (11-patient fallback split) : 0.6495
    M3  clean, official patient-disjoint     : 0.4490
    M2  clean, official patient-disjoint     : 0.4139
    published ICBHI SOTA, official 60/40     : ~0.60-0.65
per-cycle predictions -> /kaggle/working/results_M22_v2_official/preds_M22_v2_official.csv


In [13]:
# ============================================================
# CELL 13 — REQUIRED PLOTS (protocol section 5)
# ============================================================
ep = [h["epoch"] for h in history]
fig, ax = plt.subplots(2, 2, figsize=(14, 10))

ax[0, 0].plot(ep, [h["train_loss"] for h in history], "b-o", ms=3, label="train")
ax[0, 0].plot(ep, [h["val_loss"] for h in history], "r-s", ms=3, label="val")
ax[0, 0].axvline(best_epoch, color="g", ls="--", alpha=.7, label=f"best (ep {best_epoch})")
ax[0, 0].set_title("Loss"); ax[0, 0].set_xlabel("epoch"); ax[0, 0].legend(); ax[0, 0].grid(alpha=.3)

ax[0, 1].plot(ep, [h["train_accuracy"] for h in history], "b-o", ms=3, label="train acc")
ax[0, 1].plot(ep, [h["val_accuracy"] for h in history], "r-s", ms=3, label="val acc")
ax[0, 1].axvline(best_epoch, color="g", ls="--", alpha=.7)
ax[0, 1].set_title("Accuracy"); ax[0, 1].set_xlabel("epoch"); ax[0, 1].legend(); ax[0, 1].grid(alpha=.3)

ax[1, 0].plot(ep, [h["train_f1_macro"] for h in history], "b-o", ms=3, label="train F1")
ax[1, 0].plot(ep, [h["val_f1_macro"] for h in history], "r-s", ms=3, label="val F1")
ax[1, 0].plot(ep, [h["val_icbhi_score_official"] for h in history], "m-^", ms=3,
              label="val ICBHI-official")
ax[1, 0].axvline(best_epoch, color="g", ls="--", alpha=.7)
ax[1, 0].set_title("Macro-F1 and official ICBHI score")
ax[1, 0].set_xlabel("epoch"); ax[1, 0].legend(); ax[1, 0].grid(alpha=.3)

sns.heatmap(np.array(final["confusion_matrix_normalized"]), annot=True, fmt=".2f", cmap="Blues",
            xticklabels=CFG["classes"], yticklabels=CFG["classes"], ax=ax[1, 1], vmin=0, vmax=1)
ax[1, 1].set_title("Confusion matrix (row-normalised)")
ax[1, 1].set_xlabel("predicted"); ax[1, 1].set_ylabel("true")

fig.suptitle(f"{CFG['model_id']} — {CFG['model_name']}\n"
             f"official ICBHI {final['icbhi_score_official']:.4f} CI95 {ci}", fontsize=13)
fig.tight_layout()
for n in ("training_curves.png",):
    fig.savefig(os.path.join(CFG["results_dir"], n), dpi=150, bbox_inches="tight")
plt.show()

# separate files, as section 5 lists them individually
for name, key, ylab in (("loss_curve.png", ("train_loss", "val_loss"), "loss"),
                        ("accuracy_curve.png", ("train_accuracy", "val_accuracy"), "accuracy"),
                        ("f1_curve.png", ("train_f1_macro", "val_f1_macro"), "macro F1")):
    f2, a2 = plt.subplots(figsize=(7, 4.5))
    a2.plot(ep, [h[key[0]] for h in history], "b-o", ms=3, label="train")
    a2.plot(ep, [h[key[1]] for h in history], "r-s", ms=3, label="val")
    a2.axvline(best_epoch, color="g", ls="--", alpha=.7)
    a2.set_xlabel("epoch"); a2.set_ylabel(ylab); a2.legend(); a2.grid(alpha=.3)
    a2.set_title(f"{CFG['model_id']} — {ylab}")
    f2.tight_layout(); f2.savefig(os.path.join(CFG["results_dir"], name), dpi=150,
                                  bbox_inches="tight")
    plt.close(f2)

f3, a3 = plt.subplots(figsize=(6, 5))
sns.heatmap(np.array(final["confusion_matrix_raw"]), annot=True, fmt="d", cmap="Blues",
            xticklabels=CFG["classes"], yticklabels=CFG["classes"], ax=a3)
a3.set_title(f"{CFG['model_id']} — confusion matrix (raw counts)")
a3.set_xlabel("predicted"); a3.set_ylabel("true")
f3.tight_layout(); f3.savefig(os.path.join(CFG["results_dir"], "confusion_matrix.png"),
                              dpi=150, bbox_inches="tight")
plt.close(f3)
print("plots written to", CFG["results_dir"])

plots written to /kaggle/working/results_M22_v2_official


## Section 7 — Protocol-Compliant Results JSON (§4 + §4.1)

In [14]:
# ============================================================
# CELL 14 — EXPORT results_<ID>.json
# ============================================================
total_train_s = float(sum(epoch_times)) if epoch_times else float(
    sum(h.get("epoch_time_s", 0.0) for h in history))

results = {
    "meta": {
        "model_id": CFG["model_id"], "model_name": CFG["model_name"],
        "contributor": CFG["contributor"],
        "date_completed": datetime.datetime.now().strftime("%Y-%m-%d"),
        "is_augmented": bool(CFG["use_specaugment"]),
        "augmentation_method": ("SpecAugment (2 freq masks <=24 bins, 2 time masks <=80 frames), "
                                "training split only" if CFG["use_specaugment"] else "none"),
        "notes": (
            ("PUBLISHED ICBHI SPLIT, VERBATIM (539/381 recordings; 2,756 test cycles from "
             "49 test patients) so that the score is directly comparable to published work "
             "on this benchmark. NOT patient-independent: patients 156 and 218 have "
             "recordings in both train and test. Companion run to M22_v2, which is this "
             "same model, seed and schedule on the corrected patient-independent split; "
             "the two differ by the split and nothing else. " if SPLIT_MODE == "official"
             else "") +
            "Re-run of M22 on the CORRECTED official split. The original M22 executed a silent "
            "patient-id fallback (11 test patients / 492 cycles) while recording itself as the "
            "official 60/40 split (Model_Training_Protocol.md section 1). Architecture, "
            "optimiser, schedule, class weights, seed and every SpecAugment parameter are "
            "identical to that run; ONLY the split changed. Primary metric is "
            "icbhi_score_official with a patient-level bootstrap CI."),
    },
    "config": {
        "sample_rate": CFG["sample_rate"], "n_mels": CFG["n_mels"], "n_fft": CFG["n_fft"],
        "hop_length": CFG["hop_length"], "win_length": CFG["win_length"],
        "f_min": CFG["f_min"], "f_max": CFG["f_max"], "duration_s": CFG["duration_s"],
        "batch_size": CFG["batch_size"], "num_epochs": CFG["num_epochs"],
        "lr": CFG["lr"], "optimizer": "Adam", "scheduler": "CosineAnnealingLR",
        "weight_decay": CFG["weight_decay"], "grad_clip": CFG["grad_clip"],
        "architecture": f"{CFG['arch']}_pretrained_imagenet", "dropout": CFG["dropout"],
        "loss": "inverse_frequency_weighted_CrossEntropyLoss", "seed": CFG["seed"],
    },
    "environment": {
        "platform": PLATFORM, "gpu_name": GPU_NAME,
        "pytorch_version": torch.__version__, "python_version": sys.version.split()[0],
    },
    "dataset_info": {
        "dataset": "ICBHI_2017", "data_source": "real_audio",
        "train_samples": int(len(df_train)), "test_samples": int(len(df_test)),
        "train_patients": int(df_train.patient_id.nunique()),
        "test_patients": int(df_test.patient_id.nunique()),
        "train_recordings": int(c_tr), "test_recordings": int(c_te),
        "split_method": CFG["split_method"],
        "split_policy": CFG["split_policy"],
        "split_file": os.path.basename(SPLIT_FILE),
        "patient_leakage_verified": (SPLIT_MODE == "corrected"),
        "is_patient_independent": (SPLIT_MODE == "corrected"),
        "overlap_patients_reassigned_to_train": (sorted(OFFICIAL_OVERLAP)
                                                 if SPLIT_MODE == "corrected" else []),
        "overlap_patients_present_in_both": ([] if SPLIT_MODE == "corrected"
                                             else sorted(OFFICIAL_OVERLAP)),
        "comparable_to_published_icbhi_split": (SPLIT_MODE == "official"),
    },
    "efficiency": {
        "total_params": int(TOTAL_PARAMS), "trainable_params": int(TRAIN_PARAMS),
        "model_size_mb": float(MODEL_SIZE_MB),
        "training_time_total_s": round(total_train_s, 1),
        "training_time_per_epoch_s_avg": round(total_train_s / max(len(history), 1), 1),
        "gpu_name": GPU_NAME,
        "inference_time_ms_per_sample": round(inference_ms, 3),
    },
    "best_epoch": {
        "epoch": int(best_epoch), "primary_metric": "icbhi_score_official",
        "primary_metric_value": round(float(final["icbhi_score_official"]), 4),
    },
    "best_metrics": {k: (round(v, 4) if isinstance(v, float) else v)
                     for k, v in final.items()},
    "ablation": {
        "ablation_group": "augmentation_effect",
        "ablation_role": "variant" if CFG["use_specaugment"] else "baseline",
        "baseline_model_id": "M3_v2" if CFG["use_specaugment"] else None,
        "variable_changed": ("augmentation: SpecAugment (train only)"
                             if CFG["use_specaugment"] else "augmentation: none"),
        "variables_held_constant": [
            f"backbone: {CFG['arch']}_pretrained", "loss: inverse_frequency_weighted_CE",
            "optimizer: Adam", f"lr: {CFG['lr']}", "scheduler: cosine",
            f"data_split: {CFG['split_method']}", f"seed: {CFG['seed']}",
            "preprocessing: 128mel_16kHz_8s",
        ],
        "known_deviations": [
            "split policy differs from the original M22 run (reassign_to_train vs "
            "drop_from_train); the protocol section 1 policy is used here."],
        "component_flags": {
            "has_sound_event_head": True, "has_disease_head": False,
            "has_cross_task_consistency": False, "has_cqkd_regularization": False,
            "has_openmax_rejection": False, "owl_stage": 0, "compression_clusters": None,
            "has_concept_bottleneck": False, "bottleneck_type": None,
            "concept_source": None, "has_leakage_measurement": False,
            "has_concept_intervention": False, "concept_space_ood": False,
            "fm_backbone": "none",
        },
        "loss_weights": {"sound_event_weight": 1.0, "disease_weight": None,
                         "consistency_weight": None},
    },
    "training_history": history,
}

out_json = os.path.join(CFG["results_dir"], f"results_{CFG['model_id']}.json")
with open(out_json, "w") as fh:
    json.dump(results, fh, indent=2)
print(f"wrote {out_json}")
assert "confusion_matrix_raw" in results["best_metrics"], "Essential #9 violated"
assert "icbhi_score_official" in results["best_metrics"], "Essential #8 violated"
print("[OK] confusion_matrix_raw + icbhi_score_official committed.")

wrote /kaggle/working/results_M22_v2_official/results_M22_v2_official.json
[OK] confusion_matrix_raw + icbhi_score_official committed.


## Section 8 — Paired comparison: does the SpecAugment gain survive?**Run this only after both passes exist** (`VARIANT="augmented"` and `VARIANT="clean"`).Essential #10 requires a CI **and** a paired test on every headline comparison. Both models areevaluated on the identical test cycles, so predictions pair one-to-one:* **McNemar** (exact binomial) on the paired predictions, and* a **patient-level bootstrap** on the *difference* in `icbhi_score_official`.If the interval spans zero, the augmentation gain is not established on the corrected split — andthat is the answer the paper needs, whichever way it falls.

In [15]:
# ============================================================
# CELL 15 — PAIRED AUGMENTED vs CLEAN  (needs both prediction dumps)
# ============================================================
from scipy import stats

def _find_preds(mid):
    """Locate a run's per-cycle prediction dump.

    Checked in order: this run's own results dir, then Drive (where the output-setup cell
    now sends every run, including versioned re-runs like M22_v2_v2), then the legacy
    ephemeral path, then attached Kaggle datasets. Before the Drive redirect only the
    legacy path existed, so this cell reported SKIPPED even when both runs were sitting
    on Drive.
    """
    cands = []
    if isinstance(globals().get("CFG"), dict) and CFG.get("results_dir"):
        cands.append(os.path.join(CFG["results_dir"], f"preds_{mid}.csv"))
    cands += sorted(glob.glob(f"/content/drive/MyDrive/OWMTL/{mid}*/results/preds_{mid}.csv"))
    cands += sorted(glob.glob(f"/content/drive/MyDrive/OWMTL/{mid}*/preds_{mid}.csv"))
    cands.append(os.path.join(WORK, f"results_{mid}", f"preds_{mid}.csv"))
    cands += sorted(glob.glob(f"/kaggle/input/**/preds_{mid}.csv", recursive=True))
    for p in cands:
        if os.path.exists(p):
            return p
    return None

pa, pc = _find_preds("M22_v2"), _find_preds("M3_v2")
if not (pa and pc):
    print("Paired comparison SKIPPED — need both prediction dumps.")
    print(f"  augmented (M22_v2): {pa or 'MISSING'}")
    print(f"  clean     (M3_v2) : {pc or 'MISSING'}")
    print("\nRe-run this notebook with the other VARIANT, attach its output as a Kaggle\n"
          "dataset, then execute this cell. Without it there is no defensible delta.")
else:
    A, Cc = pd.read_csv(pa), pd.read_csv(pc)
    assert len(A) == len(Cc), "prediction dumps have different lengths — not the same test set"
    assert (A.y_true.values == Cc.y_true.values).all(), \
        "y_true differs between dumps — the two runs did not use the same split"

    yt = A.y_true.values
    cm_a = confusion_matrix(yt, A.y_pred.values, labels=list(range(4)))
    cm_c = confusion_matrix(yt, Cc.y_pred.values, labels=list(range(4)))
    s_a, s_c = icbhi_official(cm_a)[2], icbhi_official(cm_c)[2]

    # McNemar, exact binomial on discordant pairs
    ok_a, ok_c = A.y_pred.values == yt, Cc.y_pred.values == yt
    b = int((ok_a & ~ok_c).sum()); c_ = int((~ok_a & ok_c).sum())
    p_mc = float(stats.binomtest(min(b, c_), b + c_, 0.5).pvalue) if (b + c_) else 1.0

    # patient-level bootstrap on the DIFFERENCE
    pid = A.patient_id.values
    uq = np.unique(pid); ix = {p: np.flatnonzero(pid == p) for p in uq}
    rng2 = np.random.default_rng(SEED); diffs = []
    for _ in range(2000):
        sel = np.concatenate([ix[p] for p in rng2.choice(uq, len(uq), replace=True)])
        da = icbhi_official(confusion_matrix(yt[sel], A.y_pred.values[sel],
                                             labels=list(range(4))))[2]
        dc = icbhi_official(confusion_matrix(yt[sel], Cc.y_pred.values[sel],
                                             labels=list(range(4))))[2]
        if da == da and dc == dc:
            diffs.append(da - dc)
    lo, hi = np.percentile(diffs, [2.5, 97.5])

    print("=" * 68)
    print("SpecAugment effect on the CORRECTED official split")
    print("=" * 68)
    print(f"  M22_v2 augmented  ICBHI-official {s_a:.4f}")
    print(f"  M3_v2  clean      ICBHI-official {s_c:.4f}")
    print(f"  delta                            {s_a - s_c:+.4f}  CI95 [{lo:+.4f}, {hi:+.4f}]")
    print(f"  McNemar  b={b} c={c_}  p={p_mc:.4f}")
    verdict = ("SpecAugment HELPS (CI excludes zero)" if lo > 0 else
               "SpecAugment HURTS (CI excludes zero)" if hi < 0 else
               "NOT shown to differ — the CI spans zero. The +0.036 reported on the "
               "11-patient fallback split is NOT reproduced as a significant effect here.")
    print(f"  verdict: {verdict}")
    print("-" * 68)
    print(f"  for reference, the fallback-split claim was +0.0360 (0.6495 vs 0.6135)")
    print("=" * 68)

    with open(os.path.join(CFG["results_dir"], "augmentation_effect_corrected_split.json"),
              "w") as fh:
        json.dump({"comparison": "M22_v2 (SpecAugment) vs M3_v2 (clean)",
                   "split_method": CFG["split_method"],
                   "icbhi_score_official": {"augmented": round(s_a, 4), "clean": round(s_c, 4)},
                   "delta": round(s_a - s_c, 4),
                   "delta_ci95_patient_bootstrap": [round(float(lo), 4), round(float(hi), 4)],
                   "mcnemar": {"b": b, "c": c_, "p_exact": round(p_mc, 4)},
                   "n_test_cycles": int(len(yt)), "n_test_patients": int(len(uq)),
                   "verdict": verdict,
                   "prior_claim_on_fallback_split": 0.036}, fh, indent=2)
    print("wrote augmentation_effect_corrected_split.json")

Paired comparison SKIPPED — need both prediction dumps.
  augmented (M22_v2): MISSING
  clean     (M3_v2) : MISSING

Re-run this notebook with the other VARIANT, attach its output as a Kaggle
dataset, then execute this cell. Without it there is no defensible delta.


## Section 9 — Team Handoff Downloads (§11.D)

In [16]:
# ============================================================
# FINAL CELL — TEAM HANDOFF & ONE-CLICK FILE DOWNLOADS
# ============================================================
import os
import shutil
import glob
from IPython.display import display, FileLink

print("=" * 60)
print("OFFICIAL PROTOCOL OUTPUTS READY FOR DOWNLOAD")
print("=" * 60)

protocol_files = sorted(
    glob.glob(os.path.join(CFG["ckpt_dir"], "best_model.pth")) +
    glob.glob(os.path.join(CFG["results_dir"], "results_M*.json")) +
    glob.glob(os.path.join(CFG["results_dir"], "preds_M*.csv")) +
    glob.glob(os.path.join(CFG["results_dir"], "augmentation_effect*.json")) +
    glob.glob(os.path.join(CFG["results_dir"], "*.png"))
)

for fpath in protocol_files:
    if os.path.exists(fpath):
        size_mb = round(os.path.getsize(fpath) / (1024 * 1024), 2)
        print(f"Ready: {os.path.basename(fpath):<25} ({size_mb} MB)")
        display(FileLink(fpath))
    else:
        print(f"Missing: {os.path.basename(fpath)}")

bundle_dir = os.path.join(WORK, "protocol_bundle")
if protocol_files:
    os.makedirs(bundle_dir, exist_ok=True)
    for fpath in protocol_files:
        if os.path.exists(fpath):
            shutil.copy2(fpath, os.path.join(bundle_dir, os.path.basename(fpath)))
    zip_path = shutil.make_archive(os.path.join(WORK, f"handoff_{CFG['model_id']}"),
                                   "zip", bundle_dir)
    size_zip = round(os.path.getsize(zip_path) / (1024 * 1024), 2)
    print(f"\nOr download all official files in a single ZIP bundle ({size_zip} MB):")
    display(FileLink(zip_path))
print("=" * 60)

OFFICIAL PROTOCOL OUTPUTS READY FOR DOWNLOAD
Ready: best_model.pth            (25.88 MB)


/kaggle/working/checkpoints_M22_v2_official/best_model.pth

Ready: accuracy_curve.png        (0.05 MB)


/kaggle/working/results_M22_v2_official/accuracy_curve.png

Ready: confusion_matrix.png      (0.05 MB)


...

/kaggle/working/results_M22_v2_official/results_M22_v2_official.json

Ready: training_curves.png       (0.23 MB)


/kaggle/working/results_M22_v2_official/training_curves.png


Or download all official files in a single ZIP bundle (24.32 MB):


/kaggle/working/handoff_M22_v2_official.zip

## Section 10 — Summary & What This Result MeansFill in after both passes:| | official ICBHI | 95% CI | Se | Sp | test patients ||---|---|---|---|---|---|| original M22 (fallback split) | 0.6495 | — | 0.5696 | 0.7294 | 11 || **M22_v2** augmented | _____ | _____ | ____ | ____ | 47 || **M3_v2** clean | _____ | _____ | ____ | ____ | 47 || **Δ (augmentation)** | _____ | _____ | | | |### How to read it* **The absolute score will very likely drop.** M2 and M3 fall from ~0.61 to 0.4139 / 0.4490 when  moved onto a real patient-disjoint partition. A comparable drop here is the expected outcome, not  a failure — it is the measurement the paper is about.* **The number that matters is Δ, not the absolute.** The claim under test is *"SpecAugment gives  +0.036"*. If the Δ interval spans zero, that claim does not survive the corrected split and must  be withdrawn from the paper — the same way the single-seed bottleneck result was.* **Do not compare this to the 0.6495.** Different split, different test set, different number of  patients. The only honest comparison is M22_v2 against M3_v2 on the identical partition.### Checklist before calling this done (protocol §10)- [ ] `results_M22_v2.json` and `results_M3_v2.json` both exist- [ ] `icbhi_score_official` reported and led with; macro `icbhi_score` present but not headlined- [ ] `confusion_matrix_raw` committed in both- [ ] Bootstrap CI on each, plus McNemar + bootstrap Δ on the comparison- [ ] `split_method = official_60_40_patient_independent_corrected`, leakage assertion passed- [ ] Loss / accuracy / F1 / confusion-matrix plots saved- [ ] `python "Asif's/audit/audit_project.py"` run and clean for both IDs- [ ] `RTK_requirements.md` §6 updated — the "best model" definition currently names M22 on the      basis of a number this run replaces